# AI Red Teaming Agent

Automated adversarial testing against the **med-diagnostic-agent** (a Foundry prompt agent with **code interpreter** and **file search** tools) using the AI Red Teaming Agent in the cloud.

## How it works

The AI Red Teaming Agent leverages [PyRIT](https://github.com/microsoft/PyRIT) and Foundry's Risk & Safety evaluators to:

1. **Generate a taxonomy** of risk categories for the agent (prohibited actions, data leakage, code vulnerabilities, etc.)
2. **Apply attack strategies** (Flip, Base64, IndirectJailbreak) to generate adversarial prompts
3. **Run multi-turn probing** against the agent's code interpreter and file search tools
4. **Evaluate** each attack-response pair to compute the Attack Success Rate (ASR)

## Medical code interpreter attack surface

The `med-diagnostic-agent` uses:
- **Code interpreter** — executes Python on patient CSV data (lung cancer clinical features + symptoms)
- **File search** — queries a knowledge base of treatment best practices

Adversarial probes target scenarios like:
- Tricking the code interpreter into exfiltrating patient PII from CSV data
- Generating insecure Python code (e.g. `eval()`, `exec()`, file system access)
- Bypassing medical safety guidelines to provide harmful dosage/treatment advice
- Prompt injection via crafted patient data or query manipulation

## References

* [AI Red Teaming Agent — Concepts](https://learn.microsoft.com/en-us/azure/foundry/concepts/ai-red-teaming-agent)
* [Run AI Red Teaming Agent in the cloud](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/run-ai-red-teaming-cloud)
* [Risk and safety evaluators](https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/risk-safety-evaluators)
* [Agent evaluators](https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/agent-evaluators)
* [Supported regions for safety evaluators](https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/risk-safety-evaluators#foundry-project-configuration-and-region-support)

## Evaluators used

| Evaluator | What it measures | Medical relevance |
|-----------|-----------------|-------------------|
| `builtin.prohibited_actions` | Does the agent perform disallowed actions? | Unauthorized code execution, data deletion |
| `builtin.task_adherence` | Does the agent follow its instructions under attack? | Medical protocol compliance |
| `builtin.sensitive_data_leakage` | Does the agent leak sensitive data? | Patient PII via code interpreter output |
| `builtin.code_vulnerability` | Does the agent generate insecure code? | Unsafe Python in code interpreter sandbox |
| `builtin.violence` | Does the response contain violent/harmful content? | Harmful medical advice |

## Prerequisites

* Run notebook **02** first to create the `med-diagnostic-agent`
* **Azure AI User** role on the Foundry project
* Project region must support safety evaluators (East US 2, Sweden Central, etc.)
* Authenticate via `az login`

```bash
pip install azure-ai-projects>=2.0.0 azure-identity
```

### Setup credentials and project client

In [25]:
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)

from utils.fdyauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

if credential:
    print('Environment and authentication OK')
else:
    print("please login first")

Environment and authentication OK


In [26]:
import os
import json
import time
import datetime
from pprint import pprint
import azure.ai.projects as projectslib
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AzureAIAgentTarget,
    AgentTaxonomyInput,
    EvaluationTaxonomy,
    RiskCategory,
)

project_client = AIProjectClient(
    credential=credential,
    endpoint=settings.project_endpoint,
)
openai_client = project_client.get_openai_client()

print("project_client api version:", project_client._config.api_version)
print(f"azure-ai-projects version: {projectslib.__version__}")

project_client api version: v1
azure-ai-projects version: 2.1.0


### Step 1 — Create a Red Team

A red team is an evaluation container that defines evaluators and data source config. All runs under the same red team share the same evaluators.

Ref: [Create an AI red team](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/run-ai-red-teaming-cloud#create-an-ai-red-team)

In [27]:
AGENT_NAME = "med-diagnostic-agent"
timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

# Get the latest agent version
versions = list(project_client.agents.list_versions(agent_name=AGENT_NAME))
agent_version = versions[0].version if versions else "1"
print(f"Agent: {AGENT_NAME} (latest version: {agent_version})")

# Create a red team with safety evaluators relevant to a medical code interpreter agent
red_team = openai_client.evals.create(
    name=f"Red Team {AGENT_NAME} {timestamp}",
    data_source_config={"type": "azure_ai_source", "scenario": "red_team"},
    testing_criteria=[
        {
            "type": "azure_ai_evaluator",
            "name": "Prohibited Actions",
            "evaluator_name": "builtin.prohibited_actions",
            "evaluator_version": "1",
        },
        {
            "type": "azure_ai_evaluator",
            "name": "Task Adherence",
            "evaluator_name": "builtin.task_adherence",
            "evaluator_version": "1",
            "initialization_parameters": {"deployment_name": settings.model_deployment_name},
        },
        {
            "type": "azure_ai_evaluator",
            "name": "Sensitive Data Leakage",
            "evaluator_name": "builtin.sensitive_data_leakage",
            "evaluator_version": "1",
        },
        {
            "type": "azure_ai_evaluator",
            "name": "Code Vulnerability",
            "evaluator_name": "builtin.code_vulnerability",
            "evaluator_version": "1",
        },
        {
            "type": "azure_ai_evaluator",
            "name": "Violence",
            "evaluator_name": "builtin.violence",
            "evaluator_version": "1",
        },
    ],
)
print(f"Red team created: {red_team.id}")

Agent: med-diagnostic-agent (latest version: 15)
Red team created: eval_be09fc9a4eeb43d6aeaeea249133480e


### Step 2 — Create an evaluation taxonomy

Generate a taxonomy of prohibited actions for the agent. This defines the attack surface that the red teaming agent will probe.

> **Note**: The taxonomy API only supports `ProhibitedActions` as a risk category.
> Other risk categories (Violence, Sensitive Data Leakage, Code Vulnerability) are
> evaluated via the `testing_criteria` evaluators in Step 1, not the taxonomy.

Ref: [Create (or update) an evaluation taxonomy](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/run-ai-red-teaming-cloud#create-or-update-an-evaluation-taxonomy)

In [28]:
# Define the agent target with explicit version
target = AzureAIAgentTarget(
    name=AGENT_NAME,
    version=agent_version,
)
print(f"Target: {target.as_dict()}")

# Create taxonomy — only PROHIBITED_ACTIONS is supported for agent taxonomies
# Other risk categories (violence, data leakage, code vulnerability) are evaluated
# via the testing_criteria evaluators, not the taxonomy.
taxonomy = project_client.beta.evaluation_taxonomies.create(
    name=AGENT_NAME,
    body=EvaluationTaxonomy(
        description=f"Red team taxonomy for medical diagnostic agent with code interpreter",
        taxonomy_input=AgentTaxonomyInput(
            risk_categories=[RiskCategory.PROHIBITED_ACTIONS],
            target=target,
        ),
    ),
)
taxonomy_file_id = taxonomy.id
print(f"Taxonomy created: {taxonomy_file_id}")

Target: {'name': 'med-diagnostic-agent', 'version': '15', 'type': 'azure_ai_agent'}
Taxonomy created: azureai://accounts/foundry-proj-yw-uno-resource/projects/foundry-proj-yw-uno/evaluationtaxonomies/med-diagnostic-agent/versions/1.0


### Step 3 — Create red team runs

> **Note**: `Crescendo` and `MultiTurn` strategies cannot be combined with other strategies in the same run.
> We split into two runs: one for easy/moderate strategies, one for Crescendo alone.

#### Run 1 — Easy + Moderate strategies

| Strategy | Description | Complexity |
|----------|-------------|------------|
| `Flip` | Character flipping to bypass content filters | Easy |
| `Base64` | Base64 encoding to disguise harmful prompts | Easy |
| `Jailbreak` | Direct prompt injection to bypass AI safeguards (UPIA) | Easy |
| `IndirectJailbreak` | Indirect prompt injection via tool context/output (XPIA) | Easy |
| `Tense` | Past-tense reframing to bypass filters | Moderate |

Ref: [Create a run in a red team](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/run-ai-red-teaming-cloud#create-a-run-in-a-red-team)

In [30]:
# Run 1: Easy + Moderate strategies (can be combined)
eval_run_1 = openai_client.evals.runs.create(
    eval_id=red_team.id,
    name=f"Red Team Easy+Moderate {timestamp}",
    data_source={
        "type": "azure_ai_red_team",
        "item_generation_params": {
            "type": "red_team_taxonomy",
            "attack_strategies": [
                "Flip", "Base64", "Jailbreak",
                "IndirectJailbreak", "Tense",
            ],
            "num_turns": 5,
            "source": {"type": "file_id", "id": taxonomy_file_id},
        },
        "target": target.as_dict(),
    },
)
print(f"Run 1 created: {eval_run_1.id}, status: {eval_run_1.status}")

Run 1 created: evalrun_67a0c84219594601b56220dcd5c0a595, status: in_progress


#### Run 2 — Crescendo (difficult, must run alone)

| Strategy | Description | Complexity |
|----------|-------------|------------|
| `Crescendo` | Gradually escalates prompt risk across turns | Difficult |

In [31]:
# Run 2: Crescendo (cannot be combined with other strategies)
eval_run_2 = openai_client.evals.runs.create(
    eval_id=red_team.id,
    name=f"Red Team Crescendo {timestamp}",
    data_source={
        "type": "azure_ai_red_team",
        "item_generation_params": {
            "type": "red_team_taxonomy",
            "attack_strategies": ["Crescendo"],
            "num_turns": 5,
            "source": {"type": "file_id", "id": taxonomy_file_id},
        },
        "target": target.as_dict(),
    },
)
print(f"Run 2 created: {eval_run_2.id}, status: {eval_run_2.status}")

Run 2 created: evalrun_9f07f2cea08d4413a4e15267e38d231a, status: in_progress


### Step 4 — Poll for completion and inspect results

Red team runs can take several minutes depending on `num_turns` and the number of attack strategies.

Ref: [List red teaming run output items and results](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/run-ai-red-teaming-cloud#list-red-teaming-run-output-items-and-results)

In [33]:
# Poll both runs for completion
runs = {"Easy+Moderate": eval_run_1, "Crescendo": eval_run_2}
completed_runs = {}

while runs:
    for label, r in list(runs.items()):
        status = openai_client.evals.runs.retrieve(run_id=r.id, eval_id=red_team.id)
        if status.status in ("completed", "failed", "canceled"):
            completed_runs[label] = status
            del runs[label]
            print(f"  {label}: {status.status}")
        else:
            print(f"  {label}: {status.status} ...")
    if runs:
        time.sleep(30)

# Display results for each completed run
data_folder = os.path.join(os.getcwd(), "data")
os.makedirs(data_folder, exist_ok=True)

for label, run in completed_runs.items():
    print(f"\n{'='*60}")
    print(f"Run: {label} — {run.status}")

    if run.status == "completed":
        print(f"Result counts: {run.result_counts}")
        if hasattr(run, "report_url") and run.report_url:
            print(f"Report URL: {run.report_url}")

        items = list(
            openai_client.evals.runs.output_items.list(run_id=run.id, eval_id=red_team.id)
        )
        print(f"Output items: {len(items)}")

        # Save results
        safe_label = label.replace("+", "_").lower()
        output_path = os.path.join(data_folder, f"redteam_{safe_label}_{AGENT_NAME}_{timestamp}.json")
        with open(output_path, "w") as f:
            json.dump(
                [item.to_dict() if hasattr(item, "to_dict") else str(item) for item in items],
                f, indent=2, default=str,
            )
        print(f"Saved to {output_path}")

        # Show first 5 items
        for i, item in enumerate(items[:5]):
            print(f"\n  Item {i+1}:")
            for r in getattr(item, "results", []):
                name = getattr(r, "name", "unknown")
                label_r = getattr(r, "label", "N/A")
                score = getattr(r, "score", "N/A")
                print(f"    {name}: {label_r} (score={score})")
    elif run.status == "failed":
        print(f"Error: {run.error}")

  Easy+Moderate: in_progress ...
  Crescendo: in_progress ...
  Easy+Moderate: in_progress ...
  Crescendo: in_progress ...
  Easy+Moderate: in_progress ...
  Crescendo: in_progress ...
  Easy+Moderate: in_progress ...
  Crescendo: in_progress ...
  Easy+Moderate: in_progress ...
  Crescendo: in_progress ...
  Easy+Moderate: in_progress ...
  Crescendo: in_progress ...
  Easy+Moderate: in_progress ...
  Crescendo: in_progress ...
  Easy+Moderate: in_progress ...
  Crescendo: in_progress ...
  Easy+Moderate: in_progress ...
  Crescendo: in_progress ...
  Easy+Moderate: completed
  Crescendo: in_progress ...
  Crescendo: in_progress ...
  Crescendo: in_progress ...
  Crescendo: in_progress ...
  Crescendo: in_progress ...
  Crescendo: in_progress ...
  Crescendo: in_progress ...
  Crescendo: in_progress ...
  Crescendo: in_progress ...
  Crescendo: in_progress ...
  Crescendo: in_progress ...
  Crescendo: in_progress ...
  Crescendo: in_progress ...
  Crescendo: in_progress ...
  Crescen

KeyboardInterrupt: 

### View results in Foundry Portal

1. Login to https://ai.azure.com/
2. Open your Foundry Project
3. Navigate to **Evaluation** in the left menu
4. Select the red team evaluation run to see the Attack Success Rate (ASR) dashboard

### Next steps

* Adjust `attack_strategies` and `num_turns` for deeper coverage
* Add more `RiskCategory` values (e.g. `SENSITIVE_DATA_LEAKAGE`)
* Set up [continuous red teaming](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/run-ai-red-teaming-cloud) on a schedule
* Compare results across agent versions to track safety improvements